# GHEREMIAH AI

This is a learning course for creating a small LLM from scratch. We will follow the 8 steps of creating ML models:
1. Problem Definition
2. Data Collection
3. Data Preprocessing & Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Model Selection
7. Training & Evaluation
8. Deployment & Monitoring

## 1. Problem Definition

We want to build a small autoregressive language model that can generate text, answer chat-like prompts, and handle simple tool-use behavior in a controlled setting.

### Problem Statement
Create a compact decoder-only Transformer model from scratch that learns to predict the next token in a sequence. The model should be able to:
- generate fluent and coherent English text,
- respond in a chat-like style to short prompts,
- solve simple reasoning tasks such as basic arithmetic, common-sense questions, or short multi-step logic,
- and recognize when a prompt suggests a tool call and format a simple structured action request.

### Inputs and Outputs
- Input: a text prompt, conversation history, or task description.
- Output: the next tokens of a continuation, a helpful reply, or a structured tool-call request when appropriate.

### Functional Requirements
- The model should produce readable, grammatically reasonable text.
- It should follow instructions and stay consistent within a short context window.
- It should learn basic patterns of reasoning and tool-use behavior from synthetic or curated training data.
- It should remain small enough to train and run on a single GPU in a Colab-style environment.

### Non-Goals
- We are not trying to build a state-of-the-art general-purpose assistant.
- We are not aiming for perfect factual accuracy, long-term memory, or complex planning.
- We are not trying to connect to real external tools yet; this first version focuses on learning the pattern of tool invocation.

### Constraints
- Model size should stay small (for example, 1–8 layers and 128–512 hidden dimensions).
- Training should be feasible on modest hardware.
- The dataset should be simple, structured, and easy to understand for learning purposes.

### Success Criteria
A first version is considered successful if:
- the model produces coherent text samples after training,
- it can answer simple prompts in a natural way,
- it can complete basic reasoning-style examples with reasonable accuracy,
- and it can follow simple tool-call formatting examples when provided in the training data.

### Deliverable
A working small language model with a tokenizer, training code, and an inference script that can generate text from a prompt.

### Later Additions
- Improve reasoning quality by training on larger or more diverse data.
- Optimize for cross-platform deployment, including export to ONNX.
- Extend the model to support more complex tool calling, code understanding, and multi-step agent behavior.

## Installing Needed Resources and Packages

In [ ]:
# Installing Packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install datasets tiktoken wandb tqdm numpy

# Setting pytorch
import torch
print(torch.cuda.is_available())  # Should be True

# Mounting Drive
from google.colab import drive
drive.mount('/content/drive/GheremiahAI')

## 2. Data Collection

### Overview
For pre-training a small LLM from scratch, we need a large corpus of high-quality, coherent text. Real web-scale data (e.g., CommonCrawl) is too noisy and computationally expensive for learning/experimentation. Instead, we use curated or synthetic datasets optimized for small models.

We prioritize datasets that:
- Contain diverse, well-formed English text.
- Are small enough for rapid iteration on Colab.
- Enable quick observation of learning progress (e.g., coherent story generation).

### Datasets Used

#### Primary Dataset: TinyStories
- **Description**: A collection of short synthetic stories generated by GPT-3.5 and GPT-4. Stories use simple vocabulary (suitable for 3–4 year olds) but maintain coherent plots, grammar, and reasoning. This dataset was specifically designed to study how small language models can still produce fluent, consistent text.
- **Size**: ~2–4 million stories (train split provides hundreds of millions of tokens depending on preprocessing). The full `TinyStories_all_data.tar.gz` is ~1.6 GB compressed.
- **Source / URL**:
  - Hugging Face Dataset: [https://huggingface.co/datasets/roneneldan/TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories)
  - Direct download: [https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories_all_data.tar.gz](https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories_all_data.tar.gz)
  - Paper: [TinyStories: How Small Can Language Models Be and Still Speak Coherent English?](https://arxiv.org/abs/2305.07759)
- **Format**: 
  - Raw: `.txt` files or `.tar.gz` archive containing JSONL-like stories (one story per line or structured JSON).
  - Preprocessed: Converted to binary `.bin` files (packed token IDs as `uint16` or `int32`) or used directly via `datasets` library as text.
- **Why this format?**
  - **.txt / JSONL**: Human-readable, easy to inspect, parse, and clean. Ideal for initial EDA and custom tokenization.
  - **Binary `.bin`**: Extremely efficient for training—fast loading, low memory overhead during batching. Common in from-scratch implementations (e.g., nanoGPT style) to avoid repeated text decoding.
  - Supports streaming/large-scale processing without loading everything into RAM.

#### Secondary / Alternative Dataset: Tiny Shakespeare (for initial experiments)
- **Description**: Complete works of William Shakespeare (~1 MB raw text). Classic benchmark for character-level language models.
- **Source / URL**: Often included in nanoGPT repo examples, or downloadable from public sources like [https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) (or via Hugging Face).
- **Format**: Plain `.txt` file (raw text).
- **Why this format?**: Extremely simple for character-level tokenization (vocab ~65 chars). Perfect for debugging the full pipeline before moving to subword tokenization. Low barrier for quick experiments.

### Data Collection Process
1. **Access**: Use `datasets` library in Colab for Hugging Face datasets or `wget`/`requests` for direct downloads.
2. **License Check**: TinyStories uses CDLA-Sharing-1.0 (permissive for research/education). Verify before use.
3. **Volume**: Start with a subset (e.g., first 10–50k stories) for hyperparameter tuning, then scale to full dataset.
4. **Ethical Considerations**: Synthetic data reduces privacy/bias risks from real user text. Still, monitor for any inherited biases from the generating models.
5. **Storage**: Mount Google Drive in Colab to persist raw and processed files across sessions.

### Rationale for Dataset Choices
- **TinyStories**: Enables small models to achieve impressive coherence (as shown in the original paper). Avoids the noise of web data while still teaching real language structure. Ideal bridge between toy datasets (Shakespeare) and real pre-training.
- **Shakespeare**: Fastest way to validate the entire pipeline (tokenization → training → generation) in minutes.
- **Why not larger web datasets?** They require massive compute, heavy cleaning, and longer training—counter to our educational "from scratch" goal on Colab.

## Downloading the datasets
First, we will download tiny stories from huggingface, extract it and save it in raw datasets

In [ ]:
# CREATING RAW DATASET FOLDER

import os
from pathlib import Path

# Create directories
base_dir = Path("/content")
raw_dir = base_dir / "raw_dataset"
raw_dir.mkdir(exist_ok=True)

print(f"Raw dataset directory created at: {raw_dir}")

In [ ]:
# DOWNLOADING TINY STORIES

import requests
from tqdm import tqdm

url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories_all_data.tar.gz"
filename = raw_dir / "TinyStories_all_data.tar.gz"

print("Downloading TinyStories_all_data.tar.gz (~1.6 GB compressed)... This may take a while.")

response = requests.get(url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(filename, 'wb') as f, tqdm(
    desc=filename.name,
    total=total_size,
    unit='iB',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        size = f.write(chunk)
        bar.update(size)

print(f"\nDownload complete! File saved to: {filename}")

In [ ]:
# EXTRACT TINY MODELS

import tarfile

tar_path = raw_dir / "TinyStories_all_data.tar.gz"
extract_dir = raw_dir / "tinystories_raw"

print("Extracting TinyStories... This can take several minutes.")

with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(path=extract_dir)

print(f"Extraction complete! Files are in: {extract_dir}")

In [ ]:
# VERIFY EXTRACTED FILES

# List the extracted files
extracted_files = list(extract_dir.glob("**/*"))
print(f"Total extracted items: {len(extracted_files)}")

# Show first few files (stories are usually .txt or JSON-like)
for i, file in enumerate(extracted_files[:10]):
    print(f"{i+1}: {file.name} ({file.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# DELETING THE TAR.GZ FILE TO SAVE STORAGE AFTER ITS EXTRACTION

if tar_path.exists():
    tar_path.unlink()
    print("Compressed .tar.gz deleted to save space.")

Downloading TinyShakespeare dataset

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o raw_dataset/TinyShakespeare.txt

## 3. Data Preprocessing & Cleaning

## 4. Exploratory Data Analysis (EDA)

## 5. Feature Engineering

## 6. Model Selection

## 7. Training & Evaluation

## 8. Deployment & Monitoring